In [0]:
# Path to the uploaded file
file_path = "foodly_company_documents.txt"

# Read the entire text
with open(file_path, "r") as f:
    policy_text = f.read()
    print(policy_text)


In [0]:
from langchain_community.document_loaders import TextLoader

file_path = "foodly_company_documents.txt"
# Load the document
loader = TextLoader(file_path, encoding="utf-8")
docs = loader.load()

print(f"Loaded {len(docs)} document(s).")

print(docs[0].metadata)
print(docs[0].page_content[:1000])


In [0]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # ~800 characters per chunk
    chunk_overlap=100,   # 100 characters overlap to preserve context
    separators=["\n\n", "\n", " ", ""]

)


# "\n\n" → try to split by paragraphs first.

# "\n" → if still too big, split by lines.

# " " → if still too big, split by words.

# "" → as a last resort, split character by character.


print(f"Created {len(docs)} chunks.")



for _doc in docs:
    print(_doc)
    print("-"*100)

In [0]:
%sql

create catalog if not exists agents;
create schema if not exists agents.main;

In [0]:
# Prepare data for Delta
chunk_data = []
for i, d in enumerate(docs):
    chunk_data.append({
        "chunk_id": i + 1,
        "content": d.page_content,
        "metadata": str(d.metadata)  # store metadata as JSON-like string
    })

# Convert to dataframe

spark_df = spark.createDataFrame(chunk_data)

display(spark_df)

# Save as Delta
spark_df.write.format("delta").mode("overwrite").saveAsTable("agents.main.foodly_policy_chunks")

In [0]:
spark.sql("ALTER TABLE agents.main.foodly_policy_chunks SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

In [0]:
spark.sql(f"ALTER TABLE agents.main.foodly_policy_chunks SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 30 days')")

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

print(client.get_index(
    endpoint_name="agent-db",
    index_name="agents.main.foodly_index_name_test"
).describe())

In [0]:
#to delete vector search index
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient(disable_notice=True)

client.delete_index(
    endpoint_name="agent-db",
    index_name="agents.main.foodly_index_name_test"
)

print("Index deleted successfully")

client.delete_endpoint("agent-db")
print("Endpoint deleted successfully")

In [0]:
#to create a vector search index 
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient(disable_notice=True)

client.create_endpoint(
    name="agent-db",
    endpoint_type="STANDARD"
)

print("Endpoint creation initiated")


client.create_delta_sync_index(
    endpoint_name="agent-db",
    index_name="agents.main.foodly_index_name_test",
    source_table_name="agents.main.foodly_policy_chunks",
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="content",
    embedding_model_endpoint_name="databricks-gte-large-en"
)


### Now you can create vector serach index using this table, Follow the below doc

https://docs.databricks.com/aws/en/generative-ai/create-query-vector-search